In [20]:
from DANN.DANN_model import DANNModel
import torch
import pandas as pd
from data_loader import get_dataloaders

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = 'weights/dann_SDA_best_model.pth'
model = DANNModel(in_channels=5, num_classes= 10).to(DEVICE)
model.load_state_dict(torch.load(save_path))
model.eval()

DANNModel(
  (feature_extractor): Sequential(
    (0): Conv1d(5, 64, kernel_size=(11,), stride=(5,), padding=(5,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): ResidualTCNBlock(
      (net): Sequential(
        (0): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,))
        (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): GELU(approximate='none')
        (3): Dropout(p=0.5, inplace=False)
        (4): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,))
        (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (6): GELU(approximate='none')
        (7): Dropout(p=0.5, inplace=False)
      )
      (se): SEBlock1D(
        (squeeze): AdaptiveAvgPool1d(output_size=1)
        (excitation): Sequential(
          (0): Linear(in_features=64, out_features=4, bias=False)
          (1): ReLU()
          (

In [21]:
train_loader, val_loader, tgt_train_loader, tgt_val_loader, num_classes, le = get_dataloaders(batch_size=64)

print(le.classes_)

Loading pre-split numpy arrays from 'preprocessed/'...
Classes (10): ['barbellcurl' 'barbellrow' 'benchpress' 'bte' 'deadlift' 'dips'
 'latpulldown' 'ohp' 'pullup' 'pushup']
Loaded - Source Train: 9406, Source Val: 2300
Loaded - Target Train: 17240, Target Val: 4389
['barbellcurl' 'barbellrow' 'benchpress' 'bte' 'deadlift' 'dips'
 'latpulldown' 'ohp' 'pullup' 'pushup']


In [22]:
import numpy as np
import torch

def inference(model, loader, model_type="baseline"):
    y_true = []
    y_pred = []

    model.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            if model_type == "baseline":
                out = model(x)
            elif model_type == "coral":
                out, _ = model(x)
            elif model_type == "dann":
                out, _ = model(x)
            else:
                raise ValueError("model_type must be 'baseline', 'coral', or 'dann'")

            pred = out.argmax(dim=1)
            y_pred.extend(pred.cpu().numpy())
            y_true.extend(y.cpu().numpy())

    y_pred = np.array(y_pred)
    y_true = np.array(y_true)
    acc = (y_pred == y_true).mean()
    print("Accuracy:", acc)


In [23]:
save_path = 'weights/dann_best_model.pth'
model = DANNModel(in_channels=5, num_classes= 10).to(DEVICE)
model.load_state_dict(torch.load(save_path))
model.eval()

print("Inference results: DANN Model")
print("-"*20, "Source train", "-"*20)
inference(model, train_loader, model_type="dann")
print("-"*20, "Source validation", "-"*20)
inference(model, val_loader, model_type="dann")
print("-"*20, "Target train", "-"*20)
inference(model, tgt_train_loader, model_type="dann")
print("-"*20, "Target validation", "-"*20)
inference(model, tgt_val_loader, model_type="dann")

Inference results: DANN Model
-------------------- Source train --------------------
Accuracy: 0.995826198630137
-------------------- Source validation --------------------
Accuracy: 0.9195652173913044
-------------------- Target train --------------------
Accuracy: 0.7932736988847584
-------------------- Target validation --------------------
Accuracy: 0.7564365459102301


In [26]:
from baseline.baseline_model import AdvancedBaselineModel
import torch
import pandas as pd
from data_loader import get_dataloaders
import torch.nn as nn

class CORALWrapper(nn.Module):
    def __init__(self, base_model):
        super(CORALWrapper, self).__init__()
        self.backbone = base_model # AdvancedBaselineModel
        
    def forward(self, x):
        # 기존 모델의 구조를 활용해 특징과 분류 결과를 분리해서 반환
        x = self.backbone.stem(x)
        features = self.backbone.layers(x)
        features = features.squeeze(-1) # (Batch, 512) - 이게 CORAL이 쓰일 특징
        out = self.backbone.classifier(features)
        return out, features
    
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = 'weights/coral_best_model.pth'
base_model = AdvancedBaselineModel(in_channels=5, num_classes=num_classes)
Coral_model = CORALWrapper(base_model).to(DEVICE)

Coral_model.load_state_dict(torch.load(save_path))
Coral_model.eval()

CORALWrapper(
  (backbone): AdvancedBaselineModel(
    (stem): Sequential(
      (0): Conv1d(5, 64, kernel_size=(11,), stride=(5,), padding=(5,))
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
    )
    (layers): Sequential(
      (0): ResidualTCNBlock(
        (net): Sequential(
          (0): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,))
          (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): GELU(approximate='none')
          (3): Dropout(p=0.5, inplace=False)
          (4): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,))
          (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (6): GELU(approximate='none')
          (7): Dropout(p=0.5, inplace=False)
        )
        (se): SEBlock1D(
          (squeeze): AdaptiveAvgPool1d(output_size=1)
          (excitation): Sequential(
    

In [27]:
print("Inference results: CORAL Model")
print("-"*20, "Source train", "-"*20)
inference(Coral_model, train_loader, model_type="coral")
print("-"*20, "Source validation", "-"*20)
inference(Coral_model, val_loader, model_type="coral")
print("-"*20, "Target train", "-"*20)
inference(Coral_model, tgt_train_loader, model_type="coral")
print("-"*20, "Target validation", "-"*20)
inference(Coral_model, tgt_val_loader, model_type="coral")

Inference results: CORAL Model
-------------------- Source train --------------------
Accuracy: 0.9945419520547946
-------------------- Source validation --------------------
Accuracy: 0.9130434782608695
-------------------- Target train --------------------
Accuracy: 0.6649047397769516
-------------------- Target validation --------------------
Accuracy: 0.6637047163362952


In [32]:
save_path = 'weights/baseline_best.pth'
base_model.load_state_dict(torch.load(save_path))
base_model.eval()

AdvancedBaselineModel(
  (stem): Sequential(
    (0): Conv1d(5, 64, kernel_size=(11,), stride=(5,), padding=(5,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
  )
  (layers): Sequential(
    (0): ResidualTCNBlock(
      (net): Sequential(
        (0): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,))
        (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): GELU(approximate='none')
        (3): Dropout(p=0.5, inplace=False)
        (4): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,))
        (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (6): GELU(approximate='none')
        (7): Dropout(p=0.5, inplace=False)
      )
      (se): SEBlock1D(
        (squeeze): AdaptiveAvgPool1d(output_size=1)
        (excitation): Sequential(
          (0): Linear(in_features=64, out_features=4, bias=False)
      

In [33]:
print("Inference results: Baseline Model")
print("-"*20, "Source train", "-"*20)
inference(base_model, train_loader, model_type="baseline")
print("-"*20, "Source validation", "-"*20)
inference(base_model, val_loader, model_type="baseline")
print("-"*20, "Target train", "-"*20)
inference(base_model, tgt_train_loader, model_type="baseline")
print("-"*20, "Target validation", "-"*20)
inference(base_model, tgt_val_loader, model_type="baseline")

Inference results: Baseline Model
-------------------- Source train --------------------
Accuracy: 0.9987157534246576
-------------------- Source validation --------------------
Accuracy: 0.9404347826086956
-------------------- Target train --------------------
Accuracy: 0.5986291821561338
-------------------- Target validation --------------------
Accuracy: 0.520619731146047
